In [1]:
!pip install psycopg-binary psycopg psycopg_c asyncio


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [15]:
import urllib3.util.connection as urllib3_conn
urllib3_conn.HAS_IPV6 = False
import pandas as pd
import socket
import numpy as np
import requests
import time
from pathlib import Path
from tqdm import tqdm
import os
from pathlib import Path
import json
import asyncio
import joblib

In [17]:
print(os.getcwd())
project_root = Path.cwd().parent
damn_data = "/SA-Water-Dam-Level-Predictor/src/resources/all_dams_weekly.csv"
dam_location_data = "/SA-Water-Dam-Level-Predictor/src/resources/african_dams-African Dams.csv"
model_30 = joblib.load("/SA-Water-Dam-Level-Predictor/models/xgboost_rainfall_30.pkl")

/home/chiyedza/Downloads/Personal projects/SA-Water-Dam-Level-Predictor/notebooks


FileNotFoundError: [Errno 2] No such file or directory: '/SA-Water-Dam-Level-Predictor/models/xgboost_rainfall_30.pkl'

In [18]:
dam_df = pd.read_csv(damn_data)
dam_df.head(20)

FileNotFoundError: [Errno 2] No such file or directory: '/SA-Water-Dam-Level-Predictor/src/resources/all_dams_weekly.csv'

In [5]:
dam_location_df = pd.read_csv(dam_location_data)
dam_location_SA = dam_location_df[dam_location_df["Country"] == "South Africa"]
dam_location_SA["Name of dam"].head(50)

569          - no name -
570          - no name -
571          - no name -
572              Afgunst
573        Alartsfontein
574             Albasini
575         Albert Falls
576        Allemanskraal
577             Altenzur
578                Amcor
579           Amersfoort
580            Andalusia
581    Andrew/F C Turpin
582           Arieskraal
583            Arlington
584              Armenia
585             Aspeling
586              Athlone
587          Aucampshoop
588               Bakers
589              Balfour
590       Barend-Wessels
591        Basel Newmark
592            Beenbreek
593             Beervlei
594              Bellair
595            Ben Etive
596            Bergendal
597           Bethal (1)
598           Bethal (2)
599             Bethulie
600           Bierspruit
601                Biggs
602            Bischoffs
603    Blauwboschfontein
604         Bloemhof (1)
605         Bloemhof (2)
606              Bluegum
607                Blyde
608    Blyderivierspoort


In [6]:
filtered_dam_df = dam_df[dam_df["willDam"] != "Total"]
filtered_dam_df.info()

<class 'pandas.DataFrame'>
Index: 178 entries, 0 to 184
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   willDam     178 non-null    str    
 1   River       178 non-null    str    
 2   Photo       165 non-null    str    
 3   Indicators  178 non-null    str    
 4   FSC         178 non-null    float64
 5   This Week   178 non-null    str    
 6   Last Week   178 non-null    str    
 7   Last Year   178 non-null    str    
dtypes: float64(1), str(7)
memory usage: 12.5 KB


In [7]:
clean_name_lst = []
dam_names = filtered_dam_df["willDam"]
print(len(dam_names))

for vals in dam_names:
    name_lst = vals.split(" ")
    clean_name_lst.append(name_lst[0])
print(len(clean_name_lst))

filtered_dam_df["Clean_names"] = clean_name_lst


178
178


In [8]:
filtered_dam_df.head()

,willDam,River,Photo,Indicators,FSC,This Week,Last Week,Last Year,Clean_names
0,Bon Accord Dam,Apies River,NaN,Indicators,4.4,104.5,105.1,106.6,Bon
1,Bronkhorstspruit Dam,Bronkhorstspruit River,Photo,Indicators,57.0,100.9,101.4,102.3,Bronkhorstspruit
2,Klipdrift Dam,Loopspruit River,Photo,Indicators,13.4,#102.9,102.9,102.2,Klipdrift
3,Rietvlei Dam,Hennops River,Photo,Indicators,12.3,100.2,100.5,#100.5,Rietvlei
4,Roodeplaat Dam,Pienaars River,Photo,Indicators,41.2,100.6,100.6,100.5,Roodeplaat


In [9]:
active_dams  = dam_location_SA[dam_location_SA["Name of dam"].isin(filtered_dam_df["Clean_names"])]
filtered_active_dams = filtered_dam_df[filtered_dam_df["Clean_names"].isin(active_dams["Name of dam"])]
filtered_active_dams.head()

lat_map = active_dams.set_index("Name of dam")["Decimal degree latitude"]
long_map = active_dams.set_index("Name of dam")["Decimal degree longitude"]

filtered_active_dams["lat"] = filtered_active_dams["Clean_names"].map(lat_map)
filtered_active_dams["long"] = filtered_active_dams["Clean_names"].map(long_map)

filtered_active_dams.head()

,willDam,River,Photo,Indicators,FSC,This Week,Last Week,Last Year,Clean_names,lat,long
1,Bronkhorstspruit Dam,Bronkhorstspruit River,Photo,Indicators,57.0,100.9,101.4,102.3,Bronkhorstspruit,-25.887,28.721
4,Roodeplaat Dam,Pienaars River,Photo,Indicators,41.2,100.6,100.6,100.5,Roodeplaat,-25.621,28.373
10,Buffelskloof Dam,Waterval River,Photo,Indicators,5.3,100.5,100.6,100.6,Buffelskloof,-24.956,30.265
13,Grootdraai Dam,Vaal River,Photo,Indicators,349.8,99.9,100.0,100.9,Grootdraai,-29.300,26.917
14,Heyshope Dam,Assegaai River,Photo,Indicators,445.0,101.2,101.4,100.2,Heyshope,-27.000,30.167


In [ ]:
filtered_active_dams["lat"] = pd.to_numeric(filtered_active_dams["lat"], errors="coerce")
filtered_active_dams["long"] = pd.to_numeric(filtered_active_dams["long"], errors="coerce")


<class 'pandas.Series'>
Index: 93 entries, 1 to 183
Series name: lat
Non-Null Count  Dtype  
--------------  -----  
93 non-null     float64
dtypes: float64(1)
memory usage: 1.5 KB
